# NB_CAL_01 · Evaluar reglas de calidad y reconciliación

Lo llama `PL_MDP_MEDALLION` después de gold. Recibe en `reglas_json` la salida de `dq.usp_GET_REGLAS`, corre cada regla con Spark SQL sobre el lakehouse y devuelve los resultados con `notebook.exit`. El pipeline los registra con `dq.usp_LOG_RESULTADOS` y, si alguna regla `ERROR` falló, frena antes de refrescar el modelo semántico.

| Tipo | Qué hace | Ejemplo |
|---|---|---|
| `COMPARA` | Dos consultas con las mismas métricas (y opcionalmente una clave de cruce): falla si difieren más que la tolerancia | Filas e importe por mes en silver contra gold |
| `CONDICION` | Una consulta que devuelve un número: falla si queda fuera de `[VALOR_MIN, VALOR_MAX]` | Hechos sin dimensión = 0; horas desde la última venta ≤ 26 |

**Solo lee.** No se conecta a la base de control y no guarda filas: solo métricas y claves de cruce. Una regla que falla al ejecutarse no frena a las demás.

## Parámetros

In [ ]:
# Parámetros (celda marcada como "parameters": se pueden pasar desde un pipeline)
run_id = "manual"
reglas_json = "[]"          # salida de dq.usp_GET_REGLAS (REGLAS_JSON); para probar, pegá una lista de reglas

## Reglas

In [ ]:
# --- funciones (sin Fabric: las usa pruebas/prueba_local.py)
# Validan las reglas, comparan resultados y arman el JSON que devuelve el notebook.
# Nunca guardan filas: solo nombres de regla, claves de cruce y métricas.
import math
import re

import pandas as pd

PROHIBIDAS = r"\b(INSERT|UPDATE|DELETE|MERGE|DROP|CREATE|ALTER|TRUNCATE|GRANT|REVOKE|OPTIMIZE|VACUUM|COPY|CALL|REFRESH|CACHE|SET|USE)\b"
PATRONES_PII = ["email", "mail", "dni", "cuit", "cuil", "documento", "telefono", "celular", "direccion",
                "domicilio", "nombre", "apellido", "cbu", "tarjeta"]
MAX_DETALLE = 10   # claves que fallan que se listan en DETALLE


def _sin_comentarios(sql):
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)
    sql = re.sub(r"--[^\n]*", " ", sql)
    return re.sub(r"'(?:[^']|'')*'", "''", sql)          # los textos entre comillas no cuentan


def validar_sql(sql):
    problemas = []
    s = _sin_comentarios(str(sql or "")).strip().rstrip(";").strip()
    if not s:
        return ["SQL vacío"]
    if not re.match(r"(?is)^(SELECT|WITH)\b", s):
        problemas.append("tiene que empezar con SELECT o WITH")
    if ";" in s:
        problemas.append("una sola sentencia, sin ';' en el medio")
    m = re.search(PROHIBIDAS, s, flags=re.I)
    if m:
        problemas.append(f"no puede usar {m.group(1).upper()}: las reglas solo leen")
    return problemas


def _lista(v):
    return [x.strip() for x in str(v or "").split(",") if x.strip()]


def validar_regla(r):
    p = []
    cod = r.get("regla_code") or "?"
    if r.get("tipo") not in ("COMPARA", "CONDICION"):
        p.append(f"{cod}: tipo '{r.get('tipo')}' no válido")
    if r.get("severidad") not in ("ERROR", "WARN"):
        p.append(f"{cod}: severidad '{r.get('severidad')}' no válida")
    p += [f"{cod} sql_a: {x}" for x in validar_sql(r.get("sql_a"))]
    if r.get("tipo") == "COMPARA":
        p += [f"{cod} sql_b: {x}" for x in validar_sql(r.get("sql_b"))]
        for c in _lista(r.get("clave")):
            if any(pat in c.lower() for pat in PATRONES_PII):
                p.append(f"{cod}: la clave '{c}' parece un dato personal; cruzá por período o categoría")
    if r.get("tipo") == "CONDICION" and r.get("valor_min") is None and r.get("valor_max") is None:
        p.append(f"{cod}: falta valor_min o valor_max")
    return p


def _num(v):
    try:
        f = float(v)
        return None if math.isnan(f) else f
    except (TypeError, ValueError):
        return None


def _supera(a, b, tol_abs, tol_pct):
    """(falla, diferencia, diferencia_pct). Falla solo si supera las DOS tolerancias."""
    if a is None and b is None:
        return False, 0.0, 0.0
    if a is None or b is None:
        return True, None, None
    dif = b - a
    pct = 0.0 if dif == 0 else (math.inf if a == 0 else abs(dif) / abs(a) * 100)
    return abs(dif) > tol_abs and pct > tol_pct, dif, pct


def _fmt(v):
    if v is None:
        return "sin valor"
    return f"{v:,.0f}" if float(v).is_integer() else f"{v:,.4f}"


def resultado(regla, estado, valor_a=None, valor_b=None, dif=None, pct=None, detalle=""):
    pct = None if pct is None or math.isinf(pct) else round(pct, 4)
    return {"regla_code": regla["regla_code"], "estado": estado, "severidad": regla["severidad"],
            "valor_a": valor_a, "valor_b": valor_b, "diferencia": dif, "diferencia_pct": pct,
            "detalle": detalle[:2000]}


def comparar(regla, a: pd.DataFrame, b: pd.DataFrame):
    clave = _lista(regla.get("clave"))
    faltan = [c for c in clave if c not in a.columns or c not in b.columns]
    if faltan:
        return resultado(regla, "ERROR_EJECUCION", detalle=f"Columnas de clave que no devuelven las dos consultas: {faltan}")
    metricas = _lista(regla.get("metricas")) or [c for c in a.columns if c not in clave]
    faltan = [m for m in metricas if m not in a.columns or m not in b.columns]
    if faltan or not metricas:
        return resultado(regla, "ERROR_EJECUCION", detalle=f"Métricas que no devuelven las dos consultas: {faltan or 'ninguna'}")
    tol_abs, tol_pct = float(regla.get("tolerancia_abs") or 0), float(regla.get("tolerancia_pct") or 0)

    if clave:
        a2, b2 = a.copy(), b.copy()
        for c in clave:
            a2[c], b2[c] = a2[c].astype(str), b2[c].astype(str)
        m = a2.merge(b2, on=clave, how="outer", suffixes=("_a", "_b"), indicator=True)
    else:
        if len(a) != 1 or len(b) != 1:
            return resultado(regla, "ERROR_EJECUCION", detalle=f"Sin clave, cada consulta tiene que devolver una fila (A: {len(a)}, B: {len(b)})")
        m = a.add_suffix("_a").join(b.add_suffix("_b"))
        m["_merge"] = "both"

    fallas, peor = [], None
    for _, fila in m.iterrows():
        k = " · ".join(f"{c}={fila[c]}" for c in clave) if clave else "total"
        if fila["_merge"] != "both":
            lado = "solo en A" if fila["_merge"] == "left_only" else "solo en B"
            fallas.append(f"{k}: {lado}")
            continue
        for met in metricas:
            va, vb = _num(fila[f"{met}_a"]), _num(fila[f"{met}_b"])
            falla, dif, pct = _supera(va, vb, tol_abs, tol_pct)
            if falla:
                fallas.append(f"{k}: {met} {_fmt(va)} vs {_fmt(vb)}")
                if dif is not None and (peor is None or abs(dif) > abs(peor[2])):
                    peor = (va, vb, dif, pct)

    # Totales de la primera métrica, para el reporte
    tot_a = _num(pd.to_numeric(a[metricas[0]], errors="coerce").sum())
    tot_b = _num(pd.to_numeric(b[metricas[0]], errors="coerce").sum())
    if not fallas:
        _, dif, pct = _supera(tot_a, tot_b, 0, 0)
        return resultado(regla, "OK", tot_a, tot_b, dif, pct, f"{len(m)} claves comparadas" if clave else "")
    va, vb, dif, pct = peor if peor else (tot_a, tot_b, None, None)
    extra = f" (y {len(fallas) - MAX_DETALLE} más)" if len(fallas) > MAX_DETALLE else ""
    return resultado(regla, "FALLA", va, vb, dif, pct,
                     f"{len(fallas)} diferencias: " + " | ".join(fallas[:MAX_DETALLE]) + extra)


def condicion(regla, a: pd.DataFrame):
    if len(a) != 1 or len(a.columns) < 1:
        return resultado(regla, "ERROR_EJECUCION", detalle=f"La consulta tiene que devolver una fila con un número (devolvió {len(a)} filas)")
    v = _num(a.iloc[0, 0])
    lo, hi = _num(regla.get("valor_min")), _num(regla.get("valor_max"))
    if v is None:
        return resultado(regla, "FALLA", detalle="La consulta devolvió NULL")
    if (lo is not None and v < lo) or (hi is not None and v > hi):
        rango = f"[{_fmt(lo) if lo is not None else '-∞'}, {_fmt(hi) if hi is not None else '∞'}]"
        return resultado(regla, "FALLA", v, detalle=f"{a.columns[0]} = {_fmt(v)}, fuera de {rango}")
    return resultado(regla, "OK", v)


def evaluar_regla(regla, ejecutar):
    """ejecutar(sql) -> pandas.DataFrame. En el notebook es spark.sql(sql).toPandas()."""
    problemas = validar_regla(regla)
    if problemas:
        return resultado(regla, "ERROR_EJECUCION", detalle="Regla inválida: " + "; ".join(problemas))
    try:
        a = ejecutar(regla["sql_a"])
        if regla["tipo"] == "COMPARA":
            return comparar(regla, a, ejecutar(regla["sql_b"]))
        return condicion(regla, a)
    except Exception as e:   # noqa: BLE001 · una regla rota no frena a las demás
        msg = str(e).strip().splitlines()[0] if str(e).strip() else type(e).__name__
        return resultado(regla, "ERROR_EJECUCION", detalle=f"{type(e).__name__}: {msg[:500]}")


def evaluar(reglas, ejecutar):
    res = [evaluar_regla(r, ejecutar) for r in reglas]
    hay_error = any(r["severidad"] == "ERROR" and r["estado"] != "OK" for r in res)
    cuenta = {e: sum(r["estado"] == e for r in res) for e in ("OK", "FALLA", "ERROR_EJECUCION")}
    return {"hay_error": hay_error, "reglas": len(res), **cuenta, "resultados": res}

## Evaluar

In [ ]:
import json

reglas = json.loads(reglas_json) if reglas_json and reglas_json.strip() else []
print(f"{len(reglas)} reglas")

salida = evaluar(reglas, lambda sql: spark.sql(sql).toPandas())
salida["run_id"] = run_id

display(pd.DataFrame(salida["resultados"],
                     columns=["regla_code", "estado", "severidad", "valor_a", "valor_b", "diferencia_pct", "detalle"]))
print(f"OK {salida['OK']} · FALLA {salida['FALLA']} · ERROR_EJECUCION {salida['ERROR_EJECUCION']} · "
      f"{'FRENA' if salida['hay_error'] else 'sigue'}")

In [ ]:
# El pipeline lee esto y lo pasa a dq.usp_LOG_RESULTADOS. El notebook no falla a propósito:
# así los resultados siempre se registran y el pipeline decide con HAY_ERROR.
notebookutils.notebook.exit(json.dumps(salida, ensure_ascii=False, default=str))